# Trend Detection with HDBSCAN
HDBSCAN clustering on TF-IDF text embeddings to find trending sub-topics within the `trending_topics` dataset.

## 1. Imports & Config

In [ ]:
import os
os.makedirs("Results/HDBSCAN", exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
import hdbscan

from database_util import connect_database

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42

## 2. Load Data from PostgreSQL

In [ ]:
conn = connect_database()

cursor = conn.cursor()
cursor.execute("""
    SELECT id, text_en, topic, created_at, likes, retweets
    FROM trending_topics
    WHERE text_en IS NOT NULL
      AND TRIM(text_en) != ''
""")
rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description]
df = pd.DataFrame(rows, columns=cols)
cursor.close()
conn.close()

df["created_at"] = pd.to_datetime(df["created_at"])
df["likes"]    = df["likes"].fillna(0)
df["retweets"] = df["retweets"].fillna(0)
df["engagement"] = df["likes"] + df["retweets"]

print(f"Loaded {len(df):,} rows")
print(df["topic"].value_counts())

## 3. Text Vectorisation (TF-IDF → SVD)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True,
    stop_words="english"
)
X_tfidf = tfidf.fit_transform(df["text_en"])
print(f"TF-IDF shape: {X_tfidf.shape}")

N_COMPONENTS = 100
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(X_tfidf)
X_norm = normalize(X_svd)
print(f"SVD explained variance: {svd.explained_variance_ratio_.sum():.2%}")

## 4. HDBSCAN Clustering
Adjust MIN_CLUSTER_SIZE for clustering

In [ ]:
MIN_CLUSTER_SIZE = 30
MIN_SAMPLES      = 5

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

labels      = clusterer.fit_predict(X_norm)
soft_scores = hdbscan.all_points_membership_vectors(clusterer)

df["cluster"]       = labels
df["cluster_prob"]  = clusterer.probabilities_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()
noise_pct  = n_noise / len(labels) * 100

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise:,} ({noise_pct:.1f}%)")
print(f"Clustered posts: {len(df) - n_noise:,}")

if n_clusters > 1:
    mask  = labels != -1
    score = silhouette_score(X_norm[mask], labels[mask], metric="euclidean", sample_size=3000)
    print(f"Silhouette score (excl. noise): {score:.4f}")

print(f"Cluster labels: {sorted(set(labels) - {-1})}")

## 5. Cluster Sizes

In [ ]:
cluster_counts = (
    df[df["cluster"] != -1]
    .groupby("cluster")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

top_n = min(20, len(cluster_counts))
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=cluster_counts.head(top_n), x="cluster", y="count",
            hue="cluster", palette="tab20", legend=False, ax=ax)
ax.set_title(f"Top {top_n} HDBSCAN Clusters by Post Count")
ax.set_xlabel("Cluster ID")
ax.set_ylabel("Posts")
plt.tight_layout()
plt.savefig("Results/HDBSCAN/hdbscan_cluster_sizes.png", bbox_inches="tight")
plt.show()

## 6. Cluster Keywords (Top TF-IDF Terms)

In [ ]:
feature_names = tfidf.get_feature_names_out()
TOP_K_WORDS = 8

print("=" * 65)
print(f"{'Cluster':>8}  {'Posts':>6}  Top Keywords")
print("=" * 65)

cluster_keywords = {}
for cid in cluster_counts["cluster"]:
    mask = (df["cluster"] == cid).values
    centroid = X_tfidf[mask].mean(axis=0)
    centroid = np.asarray(centroid).flatten()
    top_idx  = centroid.argsort()[::-1][:TOP_K_WORDS]
    keywords = ", ".join(feature_names[top_idx])
    cluster_keywords[cid] = keywords
    n = mask.sum()
    print(f"{cid:>8}  {n:>6}  {keywords}")

df["cluster_keywords"] = df["cluster"].map(cluster_keywords)

## 7. Membership Confidence Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
clustered = df[df["cluster"] != -1]
ax.hist(clustered["cluster_prob"], bins=40, color="steelblue", edgecolor="white")
ax.set_title("Cluster Membership Confidence (HDBSCAN probabilities)")
ax.set_xlabel("Probability")
ax.set_ylabel("Posts")
ax.axvline(0.8, color="red", linestyle="--", label="High-confidence threshold (0.8)")
ax.legend()
plt.tight_layout()
plt.savefig("Results/HDBSCAN/hdbscan_confidence.png", bbox_inches="tight")
plt.show()

high_conf = (clustered["cluster_prob"] >= 0.8).mean() * 100
print(f"{high_conf:.1f}% of clustered posts have confidence ≥ 0.8")

## 8. Topic Distribution per Cluster

In [ ]:
top_clusters = cluster_counts.head(12)["cluster"].tolist()
pivot = (
    df[df["cluster"].isin(top_clusters)]
    .groupby(["cluster", "topic"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(13, 5))
pivot.plot(kind="bar", stacked=True, colormap="tab10", ax=ax)
ax.set_title("Original Topic Distribution per HDBSCAN Cluster")
ax.set_xlabel("Cluster ID")
ax.set_ylabel("Posts")
ax.legend(title="Topic", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Results/HDBSCAN/hdbscan_topic_distribution.png", bbox_inches="tight")
plt.show()

## 9. Engagement Analytics per Cluster

In [ ]:
eng_stats = (
    df[df["cluster"].isin(top_clusters)]
    .groupby("cluster")["engagement"]
    .agg(["mean", "median", "sum"])
    .rename(columns={"mean": "Avg Engagement", "median": "Median", "sum": "Total"})
    .sort_values("Total", ascending=False)
)

print(eng_stats.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

eng_stats["Total"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Total Engagement per Cluster")
axes[0].set_ylabel("Likes + Retweets")
axes[0].tick_params(axis="x", rotation=0)

eng_stats["Avg Engagement"].plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Average Engagement per Cluster")
axes[1].set_ylabel("Likes + Retweets")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("Results/HDBSCAN/hdbscan_engagement.png", bbox_inches="tight")
plt.show()

## 10. Cluster Activity Over Time

In [ ]:
time_df = (
    df[df["cluster"].isin(top_clusters[:6])]
    .groupby([pd.Grouper(key="created_at", freq="W"), "cluster"])
    .size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=(13, 5))
for cid, grp in time_df.groupby("cluster"):
    ax.plot(grp["created_at"], grp["count"], marker="o", label=f"Cluster {cid}")

ax.set_title("Weekly Post Volume per HDBSCAN Cluster (top 6)")
ax.set_xlabel("Week")
ax.set_ylabel("Posts")
ax.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("Results/HDBSCAN/hdbscan_time_series.png", bbox_inches="tight")
plt.show()

## 11. 2D Visualisation (SVD → 2 components)

In [ ]:
svd2 = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
X_2d = svd2.fit_transform(X_norm)

plot_df = pd.DataFrame({"x": X_2d[:, 0], "y": X_2d[:, 1],
                         "cluster": labels, "prob": clusterer.probabilities_})

noise_mask   = plot_df["cluster"] == -1
cluster_mask = ~noise_mask

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(plot_df.loc[noise_mask, "x"], plot_df.loc[noise_mask, "y"],
           c="lightgrey", s=6, alpha=0.3, label="Noise")

sc = ax.scatter(
    plot_df.loc[cluster_mask, "x"],
    plot_df.loc[cluster_mask, "y"],
    c=plot_df.loc[cluster_mask, "cluster"],
    alpha=plot_df.loc[cluster_mask, "prob"].values * 0.85 + 0.1,
    cmap="tab20", s=10
)
plt.colorbar(sc, ax=ax, label="Cluster ID")
ax.set_title("HDBSCAN Clusters — 2D SVD Projection (opacity = confidence)")
ax.set_xlabel("SVD Component 1")
ax.set_ylabel("SVD Component 2")
ax.legend()
plt.tight_layout()
plt.savefig("Results/HDBSCAN/hdbscan_2d.png", bbox_inches="tight")
plt.show()

## 12. Summary Table — Top Clusters

In [ ]:
summary = (
    df[df["cluster"] != -1]
    .groupby("cluster")
    .agg(
        posts=("id", "count"),
        avg_confidence=("cluster_prob", "mean"),
        avg_engagement=("engagement", "mean"),
        total_engagement=("engagement", "sum"),
        dominant_topic=("topic", lambda x: x.value_counts().index[0]),
        keywords=("cluster_keywords", "first")
    )
    .sort_values("posts", ascending=False)
    .head(15)
)

summary["avg_confidence"]   = summary["avg_confidence"].round(3)
summary["avg_engagement"]   = summary["avg_engagement"].round(1)
summary["total_engagement"] = summary["total_engagement"].astype(int)

print("\n── HDBSCAN Cluster Summary ──")
print(summary.to_string())

## 13. Trend Report

In [ ]:
report = (
    df[df["cluster"] != -1]
    .groupby("cluster")
    .agg(
        posts=("id", "count"),
        avg_confidence=("cluster_prob", "mean"),
        avg_engagement=("engagement", "mean"),
        total_engagement=("engagement", "sum"),
        dominant_topic=("topic", lambda x: x.value_counts().index[0]),
        first_seen=("created_at", "min"),
        last_seen=("created_at", "max"),
        keywords=("cluster_keywords", "first")
    )
    .sort_values("total_engagement", ascending=False)
    .reset_index()
)

report["avg_confidence"]  = report["avg_confidence"].round(3)
report["avg_engagement"]  = report["avg_engagement"].round(1)
report["total_engagement"] = report["total_engagement"].astype(int)

print("=" * 80)
print("EMERGING TRENDS REPORT — HDBSCAN")
print("=" * 80)
for _, row in report.iterrows():
    print(f"\n#{int(row['cluster'])} | {row['dominant_topic']}")
    print(f"   Keywords   : {row['keywords']}")
    print(f"   Posts      : {row['posts']}  |  Total Engagement: {row['total_engagement']}  |  Avg: {row['avg_engagement']}")
    print(f"   Confidence : {row['avg_confidence']}")
    print(f"   Active     : {row['first_seen'].date()} → {row['last_seen'].date()}")
print("\n" + "=" * 80)

report.to_csv("Results/HDBSCAN/trend_report_hdbscan.csv", index=False)